In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


In [ ]:
! pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset")


Skipping, found downloaded files in "./dog-and-cat-classification-dataset" (use force=True to force download)


In [ ]:
od.download("https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset")

Skipping, found downloaded files in "./microsoft-catsvsdogs-dataset" (use force=True to force download)


In [ ]:
# Generator
from pathlib import Path
project_dir = Path(r"")
train_ds=keras.utils.image_dataset_from_directory(
    directory=str(project_dir /"dog-and-cat-classification-dataset/PetImages"),
    labels="inferred",
    label_mode="int",
    class_names=None,
    batch_size=32,
    image_size=(256, 256),
    color_mode='rgb',
)
valid_ds=keras.utils.image_dataset_from_directory(
    directory=str(project_dir/"microsoft-catsvsdogs-dataset/PetImages"),
    labels="inferred",
    label_mode="int",
    class_names=None,
    batch_size=32,
    image_size=(256, 256),
    color_mode='rgb',
)

Found 24998 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [ ]:
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
valid_ds = valid_ds.apply(tf.data.experimental.ignore_errors())

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [ ]:
# Normalize
def process(image,label):
    image=tf.cast(image/255,tf.float32)
    return image,label

train_ds=train_ds.map(process)
valid_ds=valid_ds.map(process)

In [ ]:
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,297 (56.64 MB)

 Trainable params: 14,847,297 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
train_ds = train_ds.repeat()

In [ ]:
steps_per_epoch = 775
model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    steps_per_epoch=steps_per_epoch
)

Epoch 1/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 93s 120ms/step - accuracy: 0.9499 - loss: 0.1291 - val_accuracy: 0.8943 - val_loss: 0.3165
Epoch 2/10
  2/775 ━━━━━━━━━━━━━━━━━━━━ 52s 68ms/step - accuracy: 0.9141 - loss: 0.2793

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


775/775 ━━━━━━━━━━━━━━━━━━━━ 89s 115ms/step - accuracy: 0.9656 - loss: 0.0939 - val_accuracy: 0.9390 - val_loss: 0.1711
Epoch 3/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 88s 114ms/step - accuracy: 0.9765 - loss: 0.0675 - val_accuracy: 0.9359 - val_loss: 0.2014
Epoch 4/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 88s 114ms/step - accuracy: 0.9822 - loss: 0.0531 - val_accuracy: 0.9527 - val_loss: 0.1657
Epoch 5/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 88s 114ms/step - accuracy: 0.9869 - loss: 0.0392 - val_accuracy: 0.9516 - val_loss: 0.1749
Epoch 6/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 88s 114ms/step - accuracy: 0.9887 - loss: 0.0335 - val_accuracy: 0.9417 - val_loss: 0.2531
Epoch 7/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 87s 113ms/step - accuracy: 0.9886 - loss: 0.0343 - val_accuracy: 0.9721 - val_loss: 0.1021
Epoch 8/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 87s 112ms/step - accuracy: 0.9904 - loss: 0.0313 - val_accuracy: 0.9678 - val_loss: 0.1142
Epoch 9/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 144s 114ms/step - accuracy: 0.9914 - loss: 0.0273 - va

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(model, img_path):
    # Load image
    img = image.load_img(img_path, target_size=(256, 256))

    # Convert to array
    img_array = image.img_to_array(img)

    # Normalize (IMPORTANT 🔥)
    img_array = img_array / 255.0

    # Add batch dimension (1 image → batch of 1)
    img_array = np.expand_dims(img_array, axis=0)

    # Prediction
    prediction = model.predict(img_array)[0][0]

    # Interpret result
    if prediction > 0.5:
        print(f"Dog 🐶 ({prediction:.2f})")
    else:
        print(f"Cat 🐱 ({prediction:.2f})")

In [ ]:
predict_image(model,"/content/Banner_0349af78-55fb-4814-9fe7-c08abe2bdbf4.webp")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Cat 🐱 (0.00)


In [ ]:
predict_image(model,"/content/download.webp")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Dog 🐶 (1.00)


In [ ]:
predict_image(model,"/content/images.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Dog 🐶 (0.89)
